# Checking variant identifiers against the page they came from

Sylvester Kaczmarek ([@sylvesterkaczmarek](https://github.com/sylvesterkaczmarek)), KRI

An OCR engine can change one digit in an identifier and still produce a plausible string. It can also copy a correct identifier from the wrong part of a page. We will render five labelled regions from a published human eye-colour study and compare Mistral OCR with Tesseract on exactly the same pixels. Each observation records the DOI, PDF page, region, verbatim OCR line and a short image digest. A region with no identifiers is a negative control.

The unit scored is a unique **(page, region, identifier)**. The ten distinct RefSNP identifiers recur between a table and three figure captions. We score transcription and attribution only. A reader can inspect every source crop below. Expected runtime is about a minute locally, plus up to five OCR API requests if enabled.

## Set up

Use a Python notebook with network access to download the open-access paper. The local baseline needs the `tesseract` executable (5.x) with English data. On Colab or Debian, run `!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-eng` if it is missing. Set `MISTRAL_API_KEY` in the environment or Colab Secrets to enable the hosted comparison. The default without a key runs the complete local baseline and scoring checks.

In [ ]:
%pip -q install pypdfium2==5.13.0 Pillow==13.0.0

## Source and reference labels

Fan Liu et al. (2010), *Digital Quantification of Human Eye Color Highlights Genetic Association of Three New Loci*, PLOS Genetics 6(5), e1000934. [Article and licence](https://doi.org/10.1371/journal.pgen.1000934) · [Versioned PDF](https://pmc-oa-opendata.s3.amazonaws.com/PMC2865509.1/PMC2865509.1.pdf). The article is distributed under Creative Commons Attribution. We crop and rasterise the original page images, preserving the source coordinates. Labels below refer only to the visible region. They are never sent to either OCR engine.

The source PDF hash pins the coordinate system. Page numbers below start at one. Rectangles are measured in PDF points from the top-left corner. The programme uses fixed labels rather than deriving ground truth from a model or OCR result.

In [ ]:
import base64
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path
from urllib.request import Request, urlopen

import pypdfium2 as pdfium
from IPython.display import display
from PIL import Image

DOI = "10.1371/journal.pgen.1000934"
PDF_URL = "https://pmc-oa-opendata.s3.amazonaws.com/PMC2865509.1/PMC2865509.1.pdf"
PDF_HASH = "992f0f15cfaffa717a9499740deb5daa2292b383d7bb47606b820cc765cf449a"
pdf_path = Path("PMC2865509.1.pdf")
if not pdf_path.exists():
    with urlopen(PDF_URL, timeout=60) as response:
        pdf_path.write_bytes(response.read())
pdf_data = pdf_path.read_bytes()
if hashlib.sha256(pdf_data).hexdigest() != PDF_HASH:
    raise ValueError("PDF checksum differs from the labelled source. Check the cached file.")

@dataclass(frozen=True)
class Region:
    page: int
    name: str
    bounds: tuple
    labels: tuple

table_ids = ("rs3768056", "rs9782955", "rs7219915", "rs9894429", "rs12452184",
             "rs1003719", "rs2252893", "rs2835621", "rs2835630", "rs7277820")
regions = [
    Region(1, "title", (45, 40, 566, 115), ()),
    Region(6, "Table 2", (54, 449, 562, 698), table_ids),
    Region(7, "Figure 4 caption", (55, 646, 557, 694), table_ids[:2]),
    Region(8, "Figure 5 caption", (55, 701, 558, 727), table_ids[2:5]),
    Region(9, "Figure 6 caption", (56, 666, 557, 698), table_ids[5:]),
]

In [ ]:
crop_bytes = {}
scale = 3    # 216 dpi, identical for both engines
with pdfium.PdfDocument(pdf_data) as pdf:
    for region in regions:
        page = pdf[region.page - 1]
        bitmap = page.render(scale=scale)
        bounds = tuple(round(value * scale) for value in region.bounds)
        image = bitmap.to_pil().crop(bounds)
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        crop_bytes[region] = buffer.getvalue()
        print(f"Page {region.page}, {region.name}  {region.bounds}  {image.size} px")
        display(image)
        bitmap.close()
        page.close()

## Read exactly what each engine returned

The expression deliberately keeps the lowercase `rs` prefix and ASCII digits. It does no case repair or fuzzy matching. A substitution that still yields a valid-looking identifier counts as a false positive and a miss. A malformed identifier with no recognised prefix contributes a miss. Repeated mentions within one region count once. The returned line is a transcription to check against the displayed image, not independent verification.

In [ ]:
identifier = re.compile(r"(?<![A-Za-z0-9_])rs[0-9]+(?![A-Za-z0-9_])")

def observations(region, transcript):
    evidence = {}
    for line in transcript.splitlines():
        for match in identifier.finditer(line):
            key = (region.page, region.name, match.group())
            evidence.setdefault(key, {
                "doi": DOI, "page": region.page, "region": region.name, "identifier": match.group(),
                "ocr_line": line.strip(),
                "crop_sha256": hashlib.sha256(crop_bytes[region]).hexdigest(),
            })
    return evidence

if shutil.which("tesseract") is None:
    raise RuntimeError("Install tesseract-ocr and its English language data before continuing.")

print(subprocess.check_output(["tesseract", "--version"], text=True).splitlines()[0])
def baseline(data):
    result = subprocess.run(
        ["tesseract", "stdin", "stdout", "-l", "eng", "--psm", "6"],
        input=data, capture_output=True, check=True, timeout=60,
    )
    return result.stdout.decode("utf-8")

results = {"Tesseract": {}}
for region, data in crop_bytes.items():
    results["Tesseract"].update(observations(region, baseline(data)))

## Run Mistral OCR

The optional cell makes one request per cropped region. The [OCR endpoint](https://docs.mistral.ai/studio/document-processing/basic_ocr) accepts image data. Only the public crop is uploaded. The API returns one page of Markdown for each image. Inline tables keep table rows in the same transcript as other text. A failed or missing response stops the comparison and is never scored as an empty page.

In [ ]:
api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            api_key = userdata.get("MISTRAL_API_KEY")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass

def mistral_ocr(data, api_key, model):
    encoded = base64.b64encode(data).decode("ascii")
    payload = {
        "model": model, "document": {"type": "image_url", "image_url": f"data:image/png;base64,{encoded}"},
        "include_image_base64": False, "table_format": None,
    }
    request = Request("https://api.mistral.ai/v1/ocr", data=json.dumps(payload).encode(),
                      headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"})
    with urlopen(request, timeout=120) as response:
        result = json.load(response)
    pages = result["pages"]
    if len(pages) != 1 or not isinstance(pages[0].get("markdown"), str):
        raise ValueError("Expected one OCR page with a Markdown transcription.")
    return pages[0]["markdown"], result["model"]

if api_key:
    model = os.environ.get("MISTRAL_OCR_MODEL", "mistral-ocr-4-1")
    observed, served_models = {}, set()
    for region, data in crop_bytes.items():
        transcript, served_model = mistral_ocr(data, api_key, model)
        served_models.add(served_model)
        observed.update(observations(region, transcript))
    if len(served_models) != 1:
        raise ValueError(f"Different model identifiers returned across requests: {served_models}")
    results[f"Mistral ({served_models.pop()})"] = observed
else:
    print("Local baseline only. Add MISTRAL_API_KEY and rerun this cell for Mistral results.")

## Score source-grounded matches

The difference lists reveal errors a single accuracy number hides. For example, transferring a correctly spelled identifier between two regions still loses the original match and adds a false positive at the destination. The region-exact count includes the title control. Precision is undefined if an engine returns no identifiers.

In [ ]:
expected = {(r.page, r.name, label) for r in regions for label in r.labels}

def score(predictions):
    keys = set(predictions)
    matched = keys & expected
    exact_regions = sum(
        {label for page, name, label in keys if (page, name) == (r.page, r.name)} == set(r.labels)
        for r in regions
    )
    return {
        "matches": len(matched), "missing": sorted(expected - keys),
        "unexpected": sorted(keys - expected),
        "precision": len(matched) / len(keys) if keys else None,
        "recall": len(matched) / len(expected), "exact_regions_out_of_five": exact_regions,
    }

# Move one correct identifier onto the title. Its spelling alone must not earn credit.
misattributed = expected - {(7, "Figure 4 caption", "rs3768056")} | {(1, "title", "rs3768056")}
assert len(expected) == 20 and score(expected)["exact_regions_out_of_five"] == 5
assert score(misattributed)["matches"] == 19
assert len(score(misattributed)["unexpected"]) == len(score(misattributed)["missing"]) == 1
assert not observations(regions[0], "RS1, numbers 123, accession rs12x")
assert score(set())["precision"] is None

for engine, evidence in results.items():
    print(engine, json.dumps(score(evidence), indent=2))
    for key in sorted(evidence):
        row = evidence[key]
        print(f"  p{row['page']} | {row['region']} | {row['identifier']} | {row['ocr_line']}")

Five regions from one paper are enough to debug an extraction pipeline. A general performance claim would require additional papers, independent labels and a predeclared sampling rule. Selecting the region is supplied by the reference fixture here. A full-document pipeline would also need to locate each caption or table before transcription.